<a href="https://colab.research.google.com/github/lawho13/ML_Pricing/blob/main/src/tree_model_backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import ParameterGrid
import matplotlib.dates as mdates
# from sklearn.model_selection import GridSearchCV
path = "/common/home/lh811/Documents/cleaned.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load(path):
  # load csv into a pd dataframe; cast values to np.float32's for memory purposes; turn values in date to proper datetime objects; sort rows by permno, and inside permno groups, date
  df = pd.read_csv(path, dtype={col: np.float32 for col in pd.read_csv(path, nrows=0).columns if col not in ['DATE', 'year_month', 'permno']})
  df['DATE'] = pd.to_datetime(df['DATE'])
  df['year_month'] = pd.to_datetime(df['year_month']).dt.to_period('M')
  df = df.sort_values(['permno', 'DATE']).reset_index(drop=True)

  print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
  print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")

  return df


In [ ]:
clean = load(path)

Memory: 1.23 GB
Loaded 3016262 rows and 99 columns


In [ ]:
def split(df, initial_train_years=10, val_years=1, test_years=1, date_col='DATE'):

  # # sort all rows in the dataframe by their dates
  # indicator_matrix['DATE']=pd.to_datetime(indicator_matrix['DATE'], errors='coerce', format = '%Y%m%d')
  # # indicator_matrix= indicator_matrix.drop(columns=['DATE_NA', 'permno_NA'])

  # df = pd.merge(left = df,
  #               right = indicator_matrix,
  #               on =['permno', 'DATE'],
  #               how = 'left',
  #               suffixes=('', '_NA'))

  df = df.sort_values(date_col).reset_index(drop=True)

  start_date = df[date_col].min()
  end_date   = df[date_col].max()

  # let the current train_end be start_date plus the arg from the constructor for train years

  train_end = start_date + pd.DateOffset(years=initial_train_years)

  while True:
    # the backtesting loop - here, we set val_end to the date train_end + val from constructor, likewise with test_end
    val_end  = train_end + pd.DateOffset(years=val_years)
    test_end = val_end   + pd.DateOffset(years=test_years)

    # if this is the case, we've exceeded window length - break
    if test_end > end_date:
        break

    # these are all the valid data frames to return - we have a training dataframe, a validation data frame, and a test data frame - all within their respective date timelines
    train_df = df[df[date_col] <  train_end]
    val_df   = df[(df[date_col] >= train_end) & (df[date_col] < val_end)]
    test_df  = df[(df[date_col] >= val_end)   & (df[date_col] < test_end)]

    print(
        f"Train: {train_df[date_col].min().date()} → {train_df[date_col].max().date()} "
        f"({len(train_df)} rows) | "
        f"Val: {val_df[date_col].min().date()} → {val_df[date_col].max().date()} | "
        f"Test: {test_df[date_col].min().date()} → {test_df[date_col].max().date()}"
    )

    # in tuple format, return the data frames
    yield train_df, val_df, test_df

    # set train_end to test_end to adjust the window and move the time series window up for another testing round
    train_end = test_end

def get_features_and_target(df, target_col='ret'):
  drop_cols = ['DATE', 'year_month', 'permno', 'DATE_NA', 'permno_NA', target_col]
  feature_cols = [c for c in df.columns if c not in drop_cols]
  temp_df = df.dropna(subset=feature_cols + [target_col])

  X = temp_df[feature_cols].values
  y = temp_df[target_col].values

  return X, y

In [ ]:
# RF: depth, ntrees, features: paper says n_estimators=300
# GBRF: depth, ntrees, learning rate
# Training takes too long, need to make changes to get it to run.


In [ ]:
def random_forest(ntrees = 300, sample_split = 3, max_feat = 'sqrt'):

  rf_oos_results = []

  # max_depth_grid = [1, 2, 3, 4, 5, 6]

  # best_mse = float('inf')
  # best_params = None

  for train, val, test in split(clean):
    best_mse = float('inf')
    best_params = None
    best_r2 = None

    X_train, y_train = get_features_and_target(train)
    X_val, y_val  = get_features_and_target(val)
    X_test, y_test  = get_features_and_target(test)

    # for d in max_depth_grid:
    model = RandomForestRegressor(n_estimators = ntrees,
                                  min_samples_split = sample_split,
                                  max_features = max_feat,
                                  max_depth = 6, # paper
                                  n_jobs=-1,
                                  bootstrap=True
                                  )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    r2_val = r2_score(y_val, y_pred)
    mse_val = mean_squared_error(y_val, y_pred)

    if mse_val < best_mse:
      params = model.get_params()
      best_mse = mse_val
      best_r2 = r2_val
      best_params = {'n_estimators': params['n_estimators'],
                    'min_samples_split': params['min_samples_split'],
                    'max_features': params['max_features'],
                    'max_depth':params['max_depth']}

    model = RandomForestRegressor(**best_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2_test = r2_score(y_test, y_pred)
    mse_test = mean_squared_error(y_test, y_pred)

    rf_oos_results.append({
        'val_start': val['DATE'].min().date(),
        'val_end': val['DATE'].max().date(),
        'test_start': test['DATE'].min().date(),
        'test_end': test['DATE'].max().date(),
        'best_params': best_params,
        'r2_val' : best_r2,
        'mse_val': best_mse,
        'mse_test': mse_test,
        'r2_test': r2_test
    })

  return rf_oos_results

In [ ]:
rf_oos_results = random_forest()

Train: 1985-01-31 → 1994-12-30 (828888 rows) | Val: 1995-01-31 → 1995-12-29 | Test: 1996-01-31 → 1996-12-31
Train: 1985-01-31 → 1996-12-31 (1032247 rows) | Val: 1997-01-31 → 1998-01-30 | Test: 1998-02-27 → 1999-01-29
Train: 1985-01-31 → 1999-01-29 (1257329 rows) | Val: 1999-02-26 → 1999-12-31 | Test: 2000-01-31 → 2000-12-29
Train: 1985-01-31 → 2000-12-29 (1450226 rows) | Val: 2001-01-31 → 2001-12-31 | Test: 2002-01-31 → 2002-12-31
Train: 1985-01-31 → 2002-12-31 (1629936 rows) | Val: 2003-01-31 → 2004-01-30 | Test: 2004-02-27 → 2004-12-31
Train: 1985-01-31 → 2004-12-31 (1791876 rows) | Val: 2005-01-31 → 2005-12-30 | Test: 2006-01-31 → 2006-12-29
Train: 1985-01-31 → 2006-12-29 (1953636 rows) | Val: 2007-01-31 → 2007-12-31 | Test: 2008-01-31 → 2009-01-30
Train: 1985-01-31 → 2009-01-30 (2120808 rows) | Val: 2009-02-27 → 2010-01-29 | Test: 2010-02-26 → 2010-12-31
Train: 1985-01-31 → 2010-12-31 (2254126 rows) | Val: 2011-01-31 → 2011-12-30 | Test: 2012-01-31 → 2012-12-31


In [ ]:
def huber_loss(y_true, y_pred, delta):
  error = y_true - y_pred
  loss = np.where(
      np.abs(error) <= delta,
      error**2,
      2*np.abs(error)*delta - delta**2
  )
  return np.mean(loss)

In [ ]:
def gradient_boosted_tree(d = 2): # ntrees from 1-1000 but how to tune?
  gradient_boosted_oos_results = []
  gradient_boosted_h_oos_results = []

  learning_rate_grid = [0.01, 0.1]

  for train, val, test in split(clean):
    best_mse = float('inf')
    # best_params = None
    best_r2 = None
    best_learning_rate = None
    best_huber = float('inf')
    # best_r2_huber = None
    best_learning_rate_huber = None

    X_train, y_train = get_features_and_target(train)
    X_val, y_val  = get_features_and_target(val)
    X_test, y_test  = get_features_and_target(test)

    for l in learning_rate_grid:
      model = HistGradientBoostingRegressor(
          max_depth = d,
          max_iter = 1000,
          learning_rate = l,
          early_stopping=True,
          n_iter_no_change=20,
          validation_fraction=None
      )

      model.fit(X_train, y_train)
      val_pred = model.predict(X_val)
      mse_val = mean_squared_error(y_val, val_pred)

      if mse_val < best_mse:
        best_mse = mse_val
        best_r2 = r2_score(y_val, val_pred)
        best_learning_rate = l
      resid = y_train - model.predict(X_train)
      delta = np.quantile(np.abs(resid), 0.999)
      huber_loss_val = huber_loss(y_val, val_pred, delta)

      if huber_loss_val < best_huber:
        best_huber = huber_loss_val
        best_learning_rate_huber = l

    model = HistGradientBoostingRegressor(
          max_depth = d,
          max_iter = 1000,
          learning_rate = best_learning_rate,
          early_stopping=True,
          n_iter_no_change=20,
          validation_fraction=None)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2_test = r2_score(y_test, y_pred)
    mse_test = mean_squared_error(y_test, y_pred)

    model = HistGradientBoostingRegressor(
        max_depth = d,
        max_iter = 1000,
        learning_rate = best_learning_rate_huber,
        early_stopping=True,
        n_iter_no_change=20,
        validation_fraction=None)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2_test_huber = r2_score(y_test, y_pred)
    mse_test_huber = mean_squared_error(y_test, y_pred)

    gradient_boosted_oos_results.append({
        'test_start': test['DATE'].min().date(),
        'test_end': test['DATE'].max().date(),
        'best_learning_rate': best_learning_rate,
        'mse_test': mse_test,
        'r2_test': r2_test
        })

    gradient_boosted_h_oos_results.append({
        'test_start': test['DATE'].min().date(),
        'test_end': test['DATE'].max().date(),
        'best_learning_rate_huber': best_learning_rate_huber,
        'mse_test_huber': mse_test_huber,
        'r2_test_huber': r2_test_huber
    })

  return gradient_boosted_oos_results, gradient_boosted_h_oos_results

In [ ]:
gradient_boosted_oos_results, gradient_boosted_h_oos_results = gradient_boosted_tree()

In [ ]:
tree_performance = {
    'rf': rf_oos_results,
    'gradient_boosted': gradient_boosted_oos_results,
    'gradient_boosted_huber': gradient_boosted_h_oos_results
}


# ---------------------------------------------------
# Convert ALL to DataFrames safely
# ---------------------------------------------------

dfs = []

for name, res in tree_performance.items():

    df = pd.DataFrame(res)

    df['method'] = name.upper()

    dfs.append(df)

# ---------------------------------------------------
# Combine everything
# ---------------------------------------------------

all_models_mse = pd.concat(dfs, ignore_index=True)

# Ensure datetime
all_models_mse['test_start'] = pd.to_datetime(all_models_mse['test_start'])

display(all_models_mse.head())

In [ ]:
fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(14, 12),
    sharex=True
)

# ---------------------------------------------------
# Plot TEST MSE
# ---------------------------------------------------

sns.lineplot(
    data=all_models_mse,
    x='test_start',
    y='mse_test',
    hue='method',
    marker='o',
    linewidth=2,
    ax=axes[0]
)

axes[0].set_title(
    'Out-of-Sample Test MSE Across Models',
    fontsize=14,
    fontweight='bold'
)

axes[0].set_ylabel('Test MSE', fontsize=12)

axes[0].grid(True, linestyle='--', alpha=0.6)

axes[0].legend(title='Method')

# ---------------------------------------------------
# Plot TEST R²
# ---------------------------------------------------

sns.lineplot(
    data=all_models_mse,
    x='test_start',
    y='r2_test',
    hue='method',
    marker='o',
    linewidth=2,
    ax=axes[1]
)

axes[1].set_title(
    'Out-of-Sample Test R² Across Models',
    fontsize=14,
    fontweight='bold'
)

axes[1].set_xlabel('Test Window Start Date', fontsize=12)

axes[1].set_ylabel('Test R²', fontsize=12)

axes[1].grid(True, linestyle='--', alpha=0.6)

# Reference line at R² = 0
axes[1].axhline(
    y=0,
    color='red',
    linestyle='--',
    linewidth=1.5
)

axes[1].legend(title='Method')

# ---------------------------------------------------
# Format dates
# ---------------------------------------------------

axes[1].xaxis.set_major_formatter(
    mdates.DateFormatter('%Y-%b')
)

fig.autofmt_xdate()

plt.tight_layout()

In [ ]:
tree_performance.to_csv("/common/home/lh811/Documents/ML_Pricing/data/tree_models.csv", index = False)